# NB-17 — Figures rebuilt from the new results (closes the figure half of WP5; provenance for WP4)

Every figure is regenerated from files, so none is stale or hand-drawn, and each figure ends with a **provenance line** (which models, seed, split, which chips and why) that can go into the caption or the Methods. No training.

| Figure | Content | Source |
|---|---|---|
| **1** | three labelled chips: optical, radar, reference overlay (no basemap) | Sen1Floods11 rasters |
| **3** | curriculum schedule, scheduled vs the shares actually drawn | `lgmf.regime_probs` + `history.json` |
| **4** | test IoU by availability regime, per-seed dots + mean ± sd (ddof 1), all-pixel and flood-only | NB-16 per-chip table |
| **5** | qualitative predictions, **models and seed named**, chips chosen by a rule | checkpoints (seed 42) |
| **6** | accuracy against parameter count, with the SAR-only score as a second panel | `result.json` |
| **7** | failure cases **with the reference mask**, only chips that contain water | checkpoints + NB-16 per-chip table |
| **8 (new)** | paired differences with event-blocked 95 % intervals, test SAR-only | NB-16 |
| **9 (new)** | IoU against optical cloud cover, two fills, per model | NB-16 |

**Chip selection is by rule, not by eye, and every rule is printed.** If you want specific chips (for example the three that appear in the old Figure 7) set `FIG7_CHIPS`, `FIG5_CHIPS` or `FIG1_CHIPS` in the knobs cell; the rule is then skipped and the figure says "chosen by the authors".
**The text of §4.8 (failure modes) must be re-checked against the new Figure 7 by eye.** The old text described vegetation and smooth non-water surfaces; the old figure showed IoU 0.00 for chips that contain no water at all (see section 7, which lists them).

**Attach** (Add Input): `lgmf_v8` · Sen1Floods11 · `lgmf-index` · the **NB-12** and **NB-13** outputs (checkpoints and `history.json` for `seg_full_ref`, `seg_plain`, `unet_full`, `manet_full`) · the **NB-16** output (per-chip table, paired differences, cloud curves).
**Settings:** GPU T4 (CPU also works, slowly) · Save & Run All. About 5 minutes. Figures are written to `/kaggle/working/figures/` as 300 dpi PNG and PDF.

## 0. Bootstrap and knobs

In [ ]:
import sys, os, glob, json, time, shutil, warnings, re, importlib, textwrap
warnings.filterwarnings("ignore")
T_START = time.time()
for _m in [m for m in sys.modules if m == "lgmf" or m.startswith("lgmf.")]: del sys.modules[_m]
importlib.invalidate_caches()
REQUIRED = ["__init__.py", "config.py", "dataset.py", "model.py", "baselines.py", "engine.py", "curriculum.py", "metrics.py",
            "losses.py", "ema.py", "cloud.py", "ramdata.py", "audit.py", "runner.py", "selftest_v8.py"]
def _ver(p):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(p).read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)
_cands = []
for p in sorted(glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)):
    d = os.path.dirname(p); miss = [f for f in REQUIRED if not os.path.exists(os.path.join(d, f))]
    print(f"   v{'.'.join(map(str, _ver(p)))}  {'COMPLETE' if not miss else 'missing ' + ','.join(miss)}  {d}")
    if not miss: _cands.append((_ver(p), d))
assert _cands, "attach the lgmf_v8 dataset"
_best = max(_cands); assert _best[0] >= (8,), f"best complete lgmf is v{_best[0]}; attach lgmf_v8"
PKG_DIR = _best[1]; sys.path = [os.path.dirname(PKG_DIR)] + [q for q in sys.path if q != os.path.dirname(PKG_DIR)]
try: import segmentation_models_pytorch
except ImportError: os.system("pip install -q segmentation-models-pytorch")
import numpy as np, pandas as pd, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import lgmf
assert os.path.dirname(lgmf.__file__) == PKG_DIR
from lgmf.config import LGMFConfig
from lgmf.curriculum import regime_probs, eval_regimes, apply_present
from lgmf.ramdata import RamDataset
from lgmf.runner import build_from_spec
from lgmf.dataset import _read_raster
print("lgmf", lgmf.__version__)

ns = sorted(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)); assert ns, "attach lgmf-index"
idx = os.path.dirname(ns[0]); stats = json.load(open(ns[0]))
def _paths_ok(d):
    it = json.load(open(f"{d}/train_index.json"))[0]
    return all(os.path.exists(it[k]) for k in ("s1", "s2", "label") if it.get(k))
if not _paths_ok(idx):
    hits = glob.glob("/kaggle/input/**/HandLabeled", recursive=True)
    assert hits, "index paths are invalid and no 'HandLabeled' folder was found: attach the Sen1Floods11 dataset"
    where = {f: os.path.join(r, f) for r, _, fs in os.walk(hits[0]) for f in fs if f.endswith(".tif")}
    FIX = "/kaggle/working/index_fixed"; os.makedirs(FIX, exist_ok=True); lost = []
    for s in ("train", "val", "test", "bolivia"):
        items = json.load(open(f"{idx}/{s}_index.json"))
        for it in items:
            for k in ("s1", "s2", "label", "jrc"):
                if it.get(k):
                    nm = os.path.basename(it[k])
                    if nm in where: it[k] = where[nm]
                    else: lost.append((s, k, nm))
        json.dump(items, open(f"{FIX}/{s}_index.json", "w"))
    assert not lost, f"{len(lost)} image files not found, e.g. {lost[:3]}"
    idx = FIX
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)

# ------------------------------ knobs ------------------------------
EPOCHS = int(os.environ.get("NB17_EPOCHS", 45))
OUT = "/kaggle/working"; FIGDIR = f"{OUT}/figures"; os.makedirs(FIGDIR, exist_ok=True)
FIG1_CHIPS = None          # e.g. ["Ghana_1234", "India_5678", "Mekong_91011"]   None = rule
FIG5_CHIPS = None          # up to 4 test chip ids                                 None = rule
FIG7_CHIPS = None          # exactly 3 test chip ids that contain water            None = rule
QUAL_SEED = 42             # seed whose checkpoints are drawn in Figs 5 and 7
MAIN, BASE = "seg_full_ref", "seg_plain"     # the proposed SegFormer model and its no-recipe baseline (Study 1 protocol)
MIN_WATER_FRAC = 0.02      # a chip counts as 'containing water' for Figs 5 and 7 if >= 2 % of its valid pixels are water
event_of = lambda cid: cid.split("_")[0]
PROV = {}                  # figure -> provenance sentence(s)

# ------------------------------ style (validated categorical order; baselines neutral) ------------------------------
C = dict(seg="#2a78d6", unet="#eb6834", manet="#6250d6", gate="#1baf7a", base="#8a8778", ink="#0b0b0b", ink2="#52514e", grid="#e4e3df",
         tp="#2a78d6", fp="#eb6834", fn="#e87ba4", tn="#f1f0ec", nodata="#9a9a96")
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": C["ink2"], "axes.labelcolor": C["ink"],
                     "text.color": C["ink"], "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "figure.dpi": 110, "savefig.dpi": 300, "font.family": "DejaVu Sans"})
def save(fig, name):
    for ext in ("png", "pdf"): fig.savefig(f"{FIGDIR}/{name}.{ext}", bbox_inches="tight")
    plt.show(); print("saved", name)
def prov(fig, text): PROV[fig] = text; print("\nPROVENANCE", fig, ":", text)

## 1. Inputs: runs, per-chip table, NB-16 results

In [ ]:
RUNS = {}
for p in sorted(glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/*/seed*/result.json", recursive=True)):
    r = json.load(open(p)); key = (r["run_id"], int(r["seed"]))
    if key not in RUNS: RUNS[key] = os.path.dirname(p)
print(len(RUNS), "run directories;", sorted({k[0] for k in RUNS}))
for need in (MAIN, BASE): assert any(k[0] == need for k in RUNS), f"run {need} not found: attach the NB-12 output"

def find(name):
    h = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)); return h[0] if h else None
f_pc, f_st = find("nb16_perchip_all.csv.gz"), find("nb16_paired_differences.csv")
assert f_pc and f_st, "attach the NB-16 output (nb16_perchip_all.csv.gz, nb16_paired_differences.csv)"
PC = pd.read_csv(f_pc); ST = pd.read_csv(f_st)
print("per-chip table:", PC.config.nunique(), "models,", len(PC), "rows ·", f_pc)
EPS = 1e-6
def pooled(df, flood=False):
    p = "f" if flood else ""; tp, fp, fn = df[p + "tp"].sum(), df[p + "fp"].sum(), df[p + "fn"].sum(); return tp / (tp + fp + fn + EPS)
def seed_scores(config, split="test", flood=False):
    d = PC[(PC.config == config) & (PC.split == split)]
    return d.groupby(["seed", "regime"]).apply(lambda x: pooled(x, flood)).unstack()      # rows = seed, cols = regime

## 2. Figure 3 — the curriculum schedule: what was scheduled and what was actually drawn
Scheduled probabilities come from `lgmf.curriculum.regime_probs` (the training code itself); the dots are the regime shares measured in each epoch of the three `seg_full_ref` runs (`history.json`, mean over seeds).

In [ ]:
cfg = LGMFConfig(); cfg.epochs = EPOCHS; cfg.use_curriculum = True
ep = np.arange(cfg.epochs); P = np.array([regime_probs(cfg, e) for e in ep])
H = []
for sd in (42, 1, 2):
    if (MAIN, sd) in RUNS and os.path.exists(f"{RUNS[(MAIN, sd)]}/history.json"):
        h = json.load(open(f"{RUNS[(MAIN, sd)]}/history.json")); H.append(np.array([x["shares"] for x in h]))
fig, ax = plt.subplots(figsize=(3.5, 2.7))
cols = [C["base"], C["seg"], C["gate"]]; labs = ["both sensors", "SAR-only", "optical-only"]
for k in range(3):
    ax.plot(ep, P[:, k], color=cols[k], lw=1.6, label=labs[k])
    if H: ax.plot(ep[:len(np.mean(H, 0))], np.mean(H, 0)[:, k], "o", ms=2.2, color=cols[k], alpha=.55, mec="none")
ax.axvspan(0, cfg.warmup_epochs, color="#00000012", lw=0); ax.text(cfg.warmup_epochs / 2, 0.55, "warm-up", ha="center", fontsize=6.5, color=C["ink2"], rotation=90, va="center")
ax.set_xlabel("epoch"); ax.set_ylabel("sampling probability"); ax.set_ylim(-0.02, 1.02); ax.set_xlim(0, cfg.epochs - 1)
fig.legend(handles=[Line2D([], [], color=cols[k], lw=1.6, label=labs[k]) for k in range(3)] + [Line2D([], [], marker="o", ls="", color=C["ink2"], ms=3, alpha=.6, label="measured share")],
          frameon=False, loc="lower center", ncol=4, fontsize=6.5)
fig.tight_layout(rect=(0, 0.07, 1, 1)); save(fig, "fig3_schedule")
cum = P.mean(0); meas = np.mean([h.sum(0) / h.sum() for h in H], 0) if H else None
prov("Fig 3", f"Lines: lgmf.curriculum.regime_probs, {cfg.epochs} epochs, warm-up {cfg.warmup_epochs}, terminal mix {cfg.final_regime_probs}. Dots: realised shares in each epoch of {MAIN}, mean of {len(H)} seeds. "
              f"Cumulative exposure scheduled {np.round(cum, 4).tolist()}" + (f", measured {np.round(meas, 4).tolist()}." if meas is not None else "."))

## 3. Figure 4 — test IoU as sensors are removed (per-seed dots, mean ± sd with ddof = 1)
Panel (a) all valid pixels, panel (b) flood-only (JRC > 0 removed). Models trained **without** the recipe are dashed grey; models trained with it are coloured by backbone.

In [ ]:
MODELS4 = [("seg_plain", "SegFormer, no recipe", C["base"], "s", "--"), ("ref11:unet_naive", "U-Net, no recipe", "#b9b7aa", "^", "--"),
           ("unet_full", "U-Net, recipe", C["unet"], "o", "-"), ("manet_full", "MANet, recipe", C["manet"], "o", "-"), ("seg_full_ref", "SegFormer-B0, recipe (proposed)", C["seg"], "*", "-")]
MODELS4 = [m for m in MODELS4 if m[0] in set(PC.config)]
XR = [("both", "both sensors"), ("s2_only", "optical-only"), ("s1_only", "SAR-only")]
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.9), sharey=True)
for ax, flood, ttl in zip(axs, (False, True), ("(a) all valid pixels", "(b) flood-only (permanent water removed)")):
    for cfgname, lab, col, mk, ls in MODELS4:
        t = seed_scores(cfgname, "test", flood)[[r for r, _ in XR]]; x = np.arange(3)
        for _, row in t.iterrows(): ax.plot(x + np.random.RandomState(1).uniform(-.04, .04, 3), row.values, ls="", marker=mk if mk != "*" else "o", ms=2.6, color=col, alpha=.45, mec="none")
        ax.errorbar(x, t.mean().values, t.std(ddof=1).values, color=col, ls=ls, lw=1.3, marker=mk, ms=7 if mk == "*" else 4.5, capsize=2, label=lab, mec="white", mew=.5)
    ax.set_xticks(range(3)); ax.set_xticklabels([l for _, l in XR]); ax.set_title(ttl, loc="left"); ax.set_ylim(0, 0.9)
axs[0].set_ylabel("test IoU")
h_, l_ = axs[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="lower center", ncol=3, frameon=False, fontsize=6.8)
fig.tight_layout(rect=(0, 0.1, 1, 1)); save(fig, "fig4_degradation")
prov("Fig 4", "Test split (90 chips, 10 events), IoU pooled over chips; dots = individual seeds (42, 1, 2), line = mean, bars = sd (ddof = 1). Models: " +
              "; ".join(f"{l} = {c}" for c, l, *_ in MODELS4) + ". All trained with the same objective except where labelled no recipe; Study 1 protocol (ref11 = earlier runs behind the original Table 2).")

## 4. Figure 6 — accuracy against size (both-sensor and SAR-only)

In [ ]:
rows = []
for (rid, sd), d in RUNS.items():
    r = json.load(open(f"{d}/result.json")); rows.append(dict(run=rid, seed=sd, params=r["params_M"]))
PM = pd.DataFrame(rows).groupby("run").params.mean()
cand = [("seg_plain", "SegFormer-B0, no recipe", C["base"], "s"), ("unet_full", "U-Net (MobileNetV2)", C["unet"], "o"), ("manet_full", "MANet (MobileNetV2)", C["manet"], "o"),
        ("seg_full_ref", "SegFormer-B0 (proposed)", C["seg"], "*")]
cand = [c for c in cand if c[0] in PM.index and c[0] in set(PC.config)]
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.7), sharex=True)
for ax, reg, ttl in zip(axs, ("both", "s1_only"), ("(a) both sensors", "(b) SAR-only")):
    for c, lab, col, mk in cand:
        t = seed_scores(c)[reg]; ax.errorbar(PM[c], t.mean(), t.std(ddof=1), color=col, marker=mk, ms=11 if mk == "*" else 6, capsize=2, ls="", mec="white", mew=.6, label=lab)
        ax.plot([PM[c]] * len(t), t.values, ls="", marker="o", ms=2.2, color=col, alpha=.45, mec="none")
    ax.set_xscale("log"); ax.set_xlabel("parameters (millions, log scale)"); ax.set_title(ttl, loc="left")
axs[0].set_ylabel("test IoU"); h_, l_ = axs[0].get_legend_handles_labels(); fig.legend(h_, l_, frameon=False, loc="lower center", ncol=4, fontsize=6.5)
fig.tight_layout(rect=(0, 0.08, 1, 1)); save(fig, "fig6_accuracy_vs_size")
prov("Fig 6", "Test IoU (mean over 3 seeds, bars = sd with ddof 1, dots = seeds) against parameter count from result.json: " + ", ".join(f"{c} {PM[c]:.2f} M" for c, *_ in cand) + ". Front-end variants are not in this figure (see NB-14).")

## 5. Figure 8 (new) — paired differences with event-blocked intervals
Test, SAR-only IoU, A minus B. Dot = mean of per-seed differences, small dots = individual seeds, bar = 95 % interval from resampling the 10 test events. The grey band marks ±0.03, the threshold used throughout.

In [ ]:
T = ST[(ST.split == "test") & (ST.regime == "s1_only") & (ST.metric == "all-pixel")].copy()
ORD = ["RECIPE vs none, SegFormer", "RECIPE vs none, U-Net", "SegFormer minus U-Net, AS PUBLISHED", "SegFormer minus U-Net, SAME objective", "SegFormer minus MANet, SAME",
       "U-Net minus MANet", "Consistency effect, SegFormer", "Dropout alone vs consistency alone", "Schedule: curriculum minus matched", "Uniform dropout minus curriculum",
       "General model minus SAR specialist"]
def rank(c): return next((i for i, k in enumerate(ORD) if c.startswith(k)), 99)
T["o"] = T.comparison.map(rank); T = T[T.o < 99].sort_values("o").reset_index(drop=True)
SHORT = {"RECIPE vs none, SegFormer": "recipe vs none (SegFormer)", "RECIPE vs none, U-Net": "recipe vs none (U-Net)", "SegFormer minus U-Net, AS PUBLISHED": "SegFormer − U-Net, as published",
         "SegFormer minus U-Net, SAME": "SegFormer − U-Net, same objective", "SegFormer minus MANet": "SegFormer − MANet, same objective", "U-Net minus MANet": "U-Net − MANet, same objective",
         "Consistency effect": "consistency term (full − dropout only)", "Dropout alone vs consistency": "consistency only − dropout only", "Schedule": "curriculum − matched fixed mix",
         "Uniform dropout": "uniform dropout − curriculum", "General model minus SAR": "general − SAR specialist", "General model minus optical": "general − optical specialist (optical-only IoU)"}
def short(c): return next((v for k, v in SHORT.items() if c.startswith(k)), c)
fig, ax = plt.subplots(figsize=(7.0, 3.9)); y = np.arange(len(T))[::-1]
ax.axvspan(-0.03, 0.03, color="#00000010", lw=0); ax.axvline(0, color=C["ink2"], lw=.8)
for yi, r in zip(y, T.itertuples()):
    col = C["seg"] if r.event_lo > 0 else (C["unet"] if r.event_hi < 0 else C["base"])
    ax.plot([r.event_lo, r.event_hi], [yi, yi], color=col, lw=2.2, solid_capstyle="round"); ax.plot(r.mean_diff, yi, "o", ms=5.5, color=col, mec="white", mew=.6)
    ps = [float(x) for x in str(r.per_seed).split()]; ax.plot(ps, [yi] * len(ps), ls="", marker="|", ms=7, color=C["ink2"], alpha=.8)
ax.set_yticks(y); ax.set_yticklabels([short(c) for c in T.comparison], fontsize=7.2); ax.set_xlabel("difference in test SAR-only IoU (first minus second model)")
ax.grid(axis="y", visible=False)
fig.legend(handles=[Line2D([], [], marker="o", color=C["seg"], lw=2, label="interval above 0"), Line2D([], [], marker="o", color=C["unet"], lw=2, label="interval below 0"),
                   Line2D([], [], marker="o", color=C["base"], lw=2, label="interval spans 0"), Line2D([], [], marker="|", ls="", color=C["ink2"], label="single seeds"),
                   Patch(color="#00000010", label="±0.03")], frameon=False, loc="lower center", ncol=5, fontsize=6.5)
fig.tight_layout(rect=(0, 0.06, 1, 1)); save(fig, "fig8_paired_differences")
prov("Fig 8", "From NB-16 (nb16_paired_differences.csv): test split, SAR-only regime, IoU on all valid pixels; mean of per-seed differences (seeds 42, 1, 2); 95 % percentile interval from 4000 resamples of the 10 test events (same events for both models); ±0.03 is the pre-set threshold. The general-vs-optical-specialist comparison is omitted here because it is defined on optical-only IoU (see NB-16 table); this figure is SAR-only.")

## 6. Figure 9 (new) — what partial cloud does
Mean ± sd over seeds of test IoU against the fraction of optical pixels covered, for the both-sensor regime (top) and optical-only (bottom), with the two fills. Dotted lines: each model's own SAR-only IoU, i.e. what it scores if the optical image is simply ignored. The bright-cloud DN values are illustrative.

In [ ]:
curves = {r: find(f"nb16_cloud_curve_test_{r}.csv") for r in ("both", "s2_only")}
assert all(curves.values()), "NB-16 cloud curve files not found: attach the NB-16 output"
CLC = {r: pd.read_csv(f) for r, f in curves.items()}
SHOW = [("seg_plain", "SegFormer, no recipe", C["base"]), ("seg_full_ref", "SegFormer-B0, recipe", C["seg"]), ("unet_full", "U-Net, recipe", C["unet"]), ("manet_full", "MANet, recipe", C["manet"])]
SHOW = [s for s in SHOW if s[0] in set(CLC["both"].config)]
fig, ax = plt.subplots(2, 2, figsize=(7.2, 5.0), sharey=True)
for j, mode in enumerate(("zero", "bright")):
    for i, reg in enumerate(("both", "s2_only")):
        a = ax[i, j]; g = CLC[reg]; g = g[(g["mode"] == mode) | (g.cover == 0)]
        for c, lab, col in SHOW:
            m = g[g.config == c].groupby("cover").iou.agg(["mean", "std"]).fillna(0)
            a.plot(m.index * 100, m["mean"], color=col, lw=1.4, marker="o", ms=3, label=lab, ls="--" if c == "seg_plain" else "-")
            a.fill_between(m.index * 100, m["mean"] - m["std"], m["mean"] + m["std"], color=col, alpha=.14, lw=0)
            if reg == "both":
                s = seed_scores(c)["s1_only"].mean(); a.axhline(s, color=col, lw=.8, ls=":")
        a.set_title(f"{'both sensors' if reg == 'both' else 'optical-only'}, {'neutral fill (model not told)' if mode == 'zero' else 'bright cloud + shadow fill'}", loc="left", fontsize=7.5)
        a.set_xlabel("optical pixels covered (%)" if i == 1 else ""); a.set_xticks([0, 25, 50, 75, 100]); a.set_ylim(0, 0.9)
ax[0, 0].set_ylabel("test IoU"); ax[1, 0].set_ylabel("test IoU")
h_, l_ = ax[0, 0].get_legend_handles_labels(); fig.legend(h_, l_, loc="lower center", ncol=4, frameon=False, fontsize=6.8)
fig.tight_layout(rect=(0, 0.05, 1, 1)); save(fig, "fig9_partial_cloud")
prov("Fig 9", "From NB-16 (nb16_cloud_curve_test_*.csv): test split, 3 seeds, one cloud-mask draw (smooth cloud-shaped masks with exact coverage, same masks for every model); neutral fill = standardised 0 (the fill used for a missing image) with the availability flag still set; bright fill uses illustrative DN values with a dark shadow offset (lgmf/cloud.py). Dotted lines: the same model's SAR-only IoU.")

## 7. Data for Figures 1, 5 and 7: test chips, per-chip scores, and the chips with no water

In [ ]:
base = LGMFConfig(); base.s1_mean, base.s1_std, base.s2_mean, base.s2_std = stats["s1_mean"], stats["s1_std"], stats["s2_mean"], stats["s2_std"]
TEST = RamDataset(base, f"{idx}/test_index.json", augment=False)
ITEMS = {s: json.load(open(f"{idx}/{s}_index.json")) for s in ("train", "val", "test", "bolivia")}
TI = {it["id"]: it for it in ITEMS["test"]}
wf = {}
for i, cid in enumerate(TEST.ids):
    lab, val = TEST.label[i].numpy(), TEST.valid[i].numpy() > 0; wf[cid] = float((lab[val] > 0).mean()) if val.any() else 0.0
WF = pd.Series(wf, name="water_frac")
def chip_iou(config, regime, seeds=(42, 1, 2)):
    d = PC[(PC.config == config) & (PC.split == "test") & (PC.regime == regime) & (PC.seed.isin(seeds))]
    d = d.assign(iou=d.tp / (d.tp + d.fp + d.fn + EPS)); return d.groupby("id").iou.mean()
CH = pd.DataFrame({"water_frac": WF, "both": chip_iou(MAIN, "both"), "sar": chip_iou(MAIN, "s1_only"), "opt": chip_iou(MAIN, "s2_only")})
CH["event"] = CH.index.map(event_of)
NOWATER = CH[CH.water_frac == 0]
print(f"{len(NOWATER)} of {len(CH)} test chips contain NO water in the reference (per-chip IoU is 0 by construction):")
fp_pix = PC[(PC.config == MAIN) & (PC.split == "test") & (PC.regime == "s1_only") & PC.id.isin(NOWATER.index)].groupby("id").fp.mean()
print(pd.DataFrame({"event": NOWATER.event, "SAR-only false-positive pixels (mean of seeds)": fp_pix.round(0)}).to_string())
print("\nthe 12 test chips with the lowest SAR-only IoU among chips that DO contain water (>= %.0f %% water):" % (100 * MIN_WATER_FRAC))
W = CH[CH.water_frac >= MIN_WATER_FRAC].sort_values("sar"); print(W.head(12).round(3).to_string())

def show_arrays(cid):
    it = TI[cid]; s2 = _read_raster(it["s2"], list(base.s2_bands)); s1 = _read_raster(it["s1"])
    rgb = np.nan_to_num(s2[[2, 1, 0]].transpose(1, 2, 0)); rgb = np.clip(rgb / 3000.0, 0, 1) ** (1 / 1.4)
    vv = s1[0]; g = np.clip((np.nan_to_num(vv, nan=-30) + 25) / 25, 0, 1)
    return rgb, g, np.isfinite(vv)

## 8. Figure 1 — representative chips
Rule (if `FIG1_CHIPS` is None): three different events, the test chip closest to its event's median water fraction among chips with at least 5 % water. Reference water drawn in cyan over the optical composite, no basemap.

In [ ]:
if FIG1_CHIPS is None:
    sub = CH[CH.water_frac >= 0.05]; ev3 = sub.groupby("event").size().sort_values(ascending=False).index[:3]
    FIG1_CHIPS = [(sub[sub.event == e].water_frac - sub[sub.event == e].water_frac.median()).abs().idxmin() for e in ev3]; how1 = "chosen by rule (closest to the event's median water fraction)"
else: how1 = "chosen by the authors"
fig, axs = plt.subplots(len(FIG1_CHIPS), 3, figsize=(5.6, 1.9 * len(FIG1_CHIPS)))
for r, cid in enumerate(FIG1_CHIPS):
    rgb, g, ok = show_arrays(cid); i = TEST.ids.index(cid); lab = TEST.label[i].numpy() > 0
    ov = rgb.copy(); ov[lab] = 0.55 * ov[lab] + 0.45 * np.array([0.0, 0.85, 0.95])
    for a, im, t, cm in zip(axs[r], (rgb, g, ov), ("Sentinel-2 true colour", "Sentinel-1 VV", "reference water"), (None, "gray", None)):
        a.imshow(im, cmap=cm, vmin=0, vmax=1); a.set_xticks([]); a.set_yticks([]); a.grid(False)
        if r == 0: a.set_title(t, fontsize=7.5)
    axs[r, 0].set_ylabel(f"{event_of(cid)}\n{cid}", fontsize=6.5)
fig.tight_layout(); save(fig, "fig1_examples")
prov("Fig 1", f"Test-split chips {FIG1_CHIPS}, {how1}. S2 bands B4/B3/B2 scaled by 1/3000 with gamma 1/1.4; S1 VV in dB clipped to [-25, 0]; reference mask from the Sen1Floods11 hand-labelled partition (Bonafilia et al., 2020), no basemap used.")

## 9. Figures 5 and 7 — predictions from named checkpoints
Weights: the EMA weights stored in `best.pt` (the epoch selected on mean validation IoU). Fig 5 shows `seg_full_ref` in the three regimes and `seg_plain` (same backbone, no recipe) in the SAR-only regime. Colours: blue = correct water, orange = false positive, pink = missed water, pale = correct non-water, grey = no-data.

In [ ]:
def load_model(rid, sd):
    d = RUNS[(rid, sd)]; spec = json.load(open(f"{d}/config.json"))["spec"]
    m = build_from_spec(spec, pretrained=False); ck = torch.load(f"{d}/best.pt", map_location="cpu", weights_only=False)
    m.load_state_dict(ck["ema"]); return m.to(device).eval()
RG = eval_regimes()
@torch.no_grad()
def predict(model, i, reg):
    b = TEST.batch([i], "zero"); s1, s2 = b["s1"].to(device), b["s2"].to(device)
    present = RG[reg].repeat(1, 1).to(device); s1i, s2i = apply_present(s1, s2, present)
    return (torch.sigmoid(model(s1i, s2i, present)["logits"]).squeeze() > 0.5).cpu().numpy()
def err_img(pred, i):
    lab = TEST.label[i].numpy() > 0; val = TEST.valid[i].numpy() > 0
    im = np.zeros(pred.shape + (3,)); hexrgb = lambda h: np.array([int(h[k:k + 2], 16) for k in (1, 3, 5)]) / 255
    im[:] = hexrgb(C["tn"]); im[pred & lab & val] = hexrgb(C["tp"]); im[pred & ~lab & val] = hexrgb(C["fp"]); im[~pred & lab & val] = hexrgb(C["fn"]); im[~val] = hexrgb(C["nodata"])
    tp, fp, fn = (pred & lab & val).sum(), (pred & ~lab & val).sum(), (~pred & lab & val).sum()
    return im, tp / (tp + fp + fn + EPS), int(lab[val].sum())
LEG = [Patch(color=C["tp"], label="true positive"), Patch(color=C["fp"], label="false positive"), Patch(color=C["fn"], label="false negative"), Patch(color=C["nodata"], label="no data")]
M_MAIN, M_BASE = load_model(MAIN, QUAL_SEED), load_model(BASE, QUAL_SEED)

In [ ]:
# ---------------- Figure 5 ----------------
if FIG5_CHIPS is None:
    sub = CH[CH.water_frac >= 0.05]; ev4 = sub.groupby("event").size().sort_values(ascending=False).index[:4]
    FIG5_CHIPS = [(sub[sub.event == e].sar - sub[sub.event == e].sar.median()).abs().idxmin() for e in ev4]; how5 = "chosen by rule: for each of the four events with most test chips, the chip with median SAR-only IoU among chips with >= 5 % water"
else: how5 = "chosen by the authors"
cols5 = ["Sentinel-2", "Sentinel-1 VV", "reference", f"{MAIN}\nboth sensors", f"{MAIN}\noptical-only", f"{MAIN}\nSAR-only", f"{BASE}\nSAR-only (no recipe)"]
fig, axs = plt.subplots(len(FIG5_CHIPS), 7, figsize=(10.2, 1.55 * len(FIG5_CHIPS) + 0.5))
for r, cid in enumerate(FIG5_CHIPS):
    i = TEST.ids.index(cid); rgb, g, ok = show_arrays(cid); lab = TEST.label[i].numpy(); val = TEST.valid[i].numpy() > 0
    refim = np.full(lab.shape + (3,), 0.95); refim[(lab > 0) & val] = [0.16, 0.47, 0.84]; refim[~val] = [0.6, 0.6, 0.58]
    ims = [rgb, g, refim]; ttl = ["", "", f"water {100 * WF[cid]:.1f} %"]
    for model, reg in ((M_MAIN, "both"), (M_MAIN, "s2_only"), (M_MAIN, "s1_only"), (M_BASE, "s1_only")):
        im, iou, _ = err_img(predict(model, i, reg), i); ims.append(im); ttl.append(f"IoU {iou:.2f}")
    for k, a in enumerate(axs[r]):
        a.imshow(ims[k], cmap="gray" if k == 1 else None, vmin=0, vmax=1); a.set_xticks([]); a.set_yticks([]); a.grid(False)
        a.set_xlabel(ttl[k], fontsize=6.8, labelpad=2)
        if r == 0: a.set_title(cols5[k], fontsize=7)
    axs[r, 0].set_ylabel(cid, fontsize=6.3)
fig.legend(handles=LEG, loc="lower center", ncol=4, frameon=False, fontsize=7); fig.tight_layout(rect=(0, 0.04, 1, 1)); save(fig, "fig5_qualitative")
prov("Fig 5", f"Test chips {FIG5_CHIPS}, {how5}. Models: {MAIN} (SegFormer-B0, full recipe) and {BASE} (same backbone, no dropout, no consistency), seed {QUAL_SEED}, EMA weights of the epoch selected on mean validation IoU, threshold 0.5. Per-panel IoU is for that single chip.")

In [ ]:
# ---------------- Figure 7 ----------------
if FIG7_CHIPS is None:
    seen, FIG7_CHIPS = set(), []
    for cid in W.index:                                   # lowest SAR-only IoU first, one chip per event, water present
        if event_of(cid) in seen: continue
        seen.add(event_of(cid)); FIG7_CHIPS.append(cid)
        if len(FIG7_CHIPS) == 3: break
    how7 = f"chosen by rule: the three lowest SAR-only IoU (mean over 3 seeds of {MAIN}) among test chips with >= {100 * MIN_WATER_FRAC:.0f} % water, one per event"
else: how7 = "chosen by the authors"
assert all(WF[c] > 0 for c in FIG7_CHIPS), "Figure 7 must not show chips without any reference water"
fig, axs = plt.subplots(len(FIG7_CHIPS), 5, figsize=(7.6, 1.6 * len(FIG7_CHIPS) + 0.5))
for r, cid in enumerate(FIG7_CHIPS):
    i = TEST.ids.index(cid); rgb, g, ok = show_arrays(cid); lab = TEST.label[i].numpy(); val = TEST.valid[i].numpy() > 0
    refim = np.full(lab.shape + (3,), 0.95); refim[(lab > 0) & val] = [0.16, 0.47, 0.84]; refim[~val] = [0.6, 0.6, 0.58]
    imb, iob, _ = err_img(predict(M_MAIN, i, "both"), i); ims_, iou_s, _ = err_img(predict(M_MAIN, i, "s1_only"), i)
    ims = [rgb, g, refim, imb, ims_]; ttl = [event_of(cid), "", f"water {100 * WF[cid]:.1f} %", f"IoU {iob:.2f}", f"IoU {iou_s:.2f}"]
    for k, a in enumerate(axs[r]):
        a.imshow(ims[k], cmap="gray" if k == 1 else None, vmin=0, vmax=1); a.set_xticks([]); a.set_yticks([]); a.grid(False); a.set_xlabel(ttl[k], fontsize=6.8, labelpad=2)
        if r == 0: a.set_title(["Sentinel-2 (RGB)", "Sentinel-1 (VV)", "reference", "both sensors", "SAR-only"][k], fontsize=7.2)
    axs[r, 0].set_ylabel(cid, fontsize=6.3)
fig.legend(handles=LEG, loc="lower center", ncol=4, frameon=False, fontsize=7); fig.tight_layout(rect=(0, 0.04, 1, 1)); save(fig, "fig7_failure_cases")
prov("Fig 7", f"Test chips {FIG7_CHIPS}, {how7}. Model {MAIN}, seed {QUAL_SEED}, EMA weights of the selected epoch. The reference mask is shown. {len(NOWATER)} of {len(CH)} test chips have no water in the reference and are excluded (listed in section 7); per-chip IoU is 0 for them by construction.")

## 10. Manifest: what was drawn, from what, and the provenance sentences

In [ ]:
man = dict(created=time.strftime("%Y-%m-%d %H:%M"), lgmf=lgmf.__version__, epochs=EPOCHS, qual_seed=QUAL_SEED, main=MAIN, base=BASE, min_water_frac=MIN_WATER_FRAC,
           fig1=FIG1_CHIPS, fig5=FIG5_CHIPS, fig7=FIG7_CHIPS, no_water_test_chips=list(NOWATER.index), provenance=PROV,
           runs_used=sorted({f"{k[0]}/seed{k[1]}" for k in RUNS if k[0] in (MAIN, BASE, 'unet_full', 'manet_full')}))
json.dump(man, open(f"{FIGDIR}/figures_manifest.json", "w"), indent=1)
CH.round(4).to_csv(f"{FIGDIR}/test_chip_scores_main_model.csv")
for p in sorted(glob.glob(f"{FIGDIR}/*")): print(os.path.basename(p), os.path.getsize(p))
print(f"\nelapsed {(time.time()-T_START)/60:.1f} min")